# ECG Project — Notebook 3: Batch PDF Dataset Builder

This notebook converts a ZIP containing ECG PDF reports into a clean tabular dataset.

## Main tasks

1. Upload one ZIP containing all ECG PDFs.
2. Extract embedded PDF text.
3. Use OCR only when embedded text is unavailable.
4. Parse ECG measurements and machine findings.
5. Detect exact duplicate PDFs.
6. Detect repeated case IDs from filenames.
7. Validate extracted values.
8. Create leakage-safe train, validation, and test splits.
9. Save CSV, JSON, failure logs, and split manifests.

> Keep all original reports together in one ZIP. Do not manually split the PDFs.


## 1. Install dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr
!pip install -q pypdf pdf2image pytesseract pandas numpy scikit-learn tqdm


## 2. Upload the ECG ZIP file

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Upload one ZIP file containing the ECG PDFs.")

zip_path = Path(zip_files[0])
print("Selected ZIP:", zip_path)


## 3. Extract the ZIP and locate all PDFs

In [ ]:
import zipfile
import shutil
from pathlib import Path

extract_dir = Path("/content/ecg_reports")

if extract_dir.exists():
    shutil.rmtree(extract_dir)

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_dir)

pdf_paths = sorted(
    path for path in extract_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
)

print("PDF reports found:", len(pdf_paths))
print("\nFirst 10 files:")
for path in pdf_paths[:10]:
    print("-", path.relative_to(extract_dir))


## 4. Define text extraction and ECG parsing functions

Because the reports come from one hospital and one report format, this parser is intentionally optimized for that format.

Embedded text is much faster and cleaner than OCR, so OCR is used only as a fallback.


In [ ]:
import re
import hashlib
from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract

def calculate_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def extract_embedded_text(path):
    reader = PdfReader(str(path))
    return "\n".join((page.extract_text() or "") for page in reader.pages).strip()

def extract_ocr_text(path, dpi=250):
    images = convert_from_path(str(path), dpi=dpi)
    return "\n".join(
        pytesseract.image_to_string(image, config="--psm 6")
        for image in images
    ).strip()

def extract_report_text(path):
    embedded = extract_embedded_text(path)

    if len(embedded) >= 80:
        return embedded, "embedded_pdf_text"

    ocr_text = extract_ocr_text(path)
    return ocr_text, "ocr_fallback"

def normalize_text(text):
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00b0", " ° ")
    text = re.sub(r"[\t\r]+", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

def first_match(pattern, text, cast=float, flags=re.IGNORECASE):
    match = re.search(pattern, text, flags)
    if not match:
        return None

    try:
        return cast(match.group(1))
    except (ValueError, TypeError):
        return None

def parse_qt_qtc(text):
    match = re.search(
        r"QT\s*/\s*QTc(?:Bz)?\s*[:\-]?\s*"
        r"(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)\s*ms",
        text,
        re.IGNORECASE
    )
    if not match:
        return None, None

    return float(match.group(1)), float(match.group(2))

def parse_axes(text):
    match = re.search(
        r"P\s*/\s*QRS\s*/\s*T\s*[:\-]?\s*"
        r"([+-]?\d+(?:\.\d+)?)\s*/\s*"
        r"([+-]?\d+(?:\.\d+)?)\s*/\s*"
        r"([+-]?\d+(?:\.\d+)?)",
        text,
        re.IGNORECASE
    )
    if not match:
        return None, None, None

    return tuple(float(match.group(index)) for index in range(1, 4))

def parse_rv5_sv1(text):
    match = re.search(
        r"RV5\s*/\s*SV1\s*[:\-]?\s*"
        r"([+-]?\d+(?:\.\d+)?)\s*/\s*"
        r"([+-]?\d+(?:\.\d+)?)\s*mV",
        text,
        re.IGNORECASE
    )
    if not match:
        return None, None

    return float(match.group(1)), float(match.group(2))

def extract_diagnosis_lines(text):
    patterns = [
        r"Diagnosis\s+Information\s*:\s*(.*?)(?:Report\s+Confirmed\s+by|$)",
        r"Diagnosis\s*:\s*(.*?)(?:Report\s+Confirmed\s+by|$)"
    ]

    block = None
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            block = match.group(1)
            break

    if not block:
        return []

    lines = []
    for raw_line in block.splitlines():
        line = re.sub(r"\s+", " ", raw_line).strip(" :-")
        if not line:
            continue

        # Avoid common footer/header fragments.
        if re.search(
            r"(patient|hospital|doctor|confirmed|printed|page\s+\d+)",
            line,
            re.IGNORECASE
        ):
            continue

        if len(line) > 2:
            lines.append(line)

    # Preserve order while removing duplicates.
    return list(dict.fromkeys(lines))

def infer_case_id(filename):
    stem = Path(filename).stem

    # Examples such as 20260304-195122-64.pdf -> case ID 64
    match = re.search(r"-(\d+)$", stem)
    if match:
        return match.group(1)

    return stem

def parse_ecg_report(text, extraction_method):
    text = normalize_text(text)

    qt_ms, qtc_ms = parse_qt_qtc(text)
    p_axis, qrs_axis, t_axis = parse_axes(text)
    rv5_mv, sv1_mv = parse_rv5_sv1(text)
    diagnosis_lines = extract_diagnosis_lines(text)

    rhythm = None
    findings = []

    for line in diagnosis_lines:
        if rhythm is None and re.search(r"\brhythm\b", line, re.IGNORECASE):
            rhythm = line
        else:
            findings.append(line)

    if findings:
        primary_label = findings[0]
    elif rhythm:
        primary_label = rhythm
    else:
        primary_label = None

    return {
        "heart_rate_bpm": first_match(
            r"\bHR\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*bpm", text
        ),
        "p_duration_ms": first_match(
            r"(?m)^\s*P\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*ms", text
        ),
        "pr_interval_ms": first_match(
            r"\bPR\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*ms", text
        ),
        "qrs_duration_ms": first_match(
            r"\bQRS\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*ms", text
        ),
        "qt_interval_ms": qt_ms,
        "qtc_bazett_ms": qtc_ms,
        "p_axis_deg": p_axis,
        "qrs_axis_deg": qrs_axis,
        "t_axis_deg": t_axis,
        "rv5_mv": rv5_mv,
        "sv1_mv": sv1_mv,
        "rhythm": rhythm,
        "machine_findings": findings,
        "primary_label": primary_label,
        "paper_speed_mm_s": first_match(
            r"(\d+(?:\.\d+)?)\s*mm/s", text
        ),
        "gain_mm_mv": first_match(
            r"(\d+(?:\.\d+)?)\s*mm/mV", text
        ),
        "report_datetime": first_match(
            r"(\d{2}[-/]\d{2}[-/]\d{4}\s+"
            r"\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM)?)",
            text,
            cast=str
        ),
        "extraction_method": extraction_method
    }


## 5. Define validation rules

In [ ]:
FEATURE_COLUMNS = [
    "heart_rate_bpm",
    "p_duration_ms",
    "pr_interval_ms",
    "qrs_duration_ms",
    "qt_interval_ms",
    "qtc_bazett_ms",
    "p_axis_deg",
    "qrs_axis_deg",
    "t_axis_deg",
    "rv5_mv",
    "sv1_mv"
]

VALID_RANGES = {
    "heart_rate_bpm": (20, 250),
    "p_duration_ms": (40, 200),
    "pr_interval_ms": (60, 400),
    "qrs_duration_ms": (40, 250),
    "qt_interval_ms": (150, 700),
    "qtc_bazett_ms": (150, 700),
    "p_axis_deg": (-180, 180),
    "qrs_axis_deg": (-180, 180),
    "t_axis_deg": (-180, 180),
    "rv5_mv": (0, 10),
    "sv1_mv": (0, 10),
    "paper_speed_mm_s": (10, 100),
    "gain_mm_mv": (2, 40)
}

def validate_row(row):
    warnings = []

    for field, (minimum, maximum) in VALID_RANGES.items():
        value = row.get(field)

        if value is None:
            warnings.append(f"missing:{field}")
        elif not minimum <= value <= maximum:
            warnings.append(f"out_of_range:{field}={value}")

    if not row.get("primary_label"):
        warnings.append("missing:primary_label")

    missing_features = sum(row.get(column) is None for column in FEATURE_COLUMNS)
    row["missing_feature_count"] = missing_features
    row["feature_completeness_pct"] = round(
        100 * (len(FEATURE_COLUMNS) - missing_features) / len(FEATURE_COLUMNS),
        2
    )

    row["validation_warnings"] = warnings
    row["is_model_ready"] = (
        missing_features == 0
        and row.get("primary_label") is not None
    )

    return row


## 6. Process all ECG reports

For a few hundred reports, embedded text extraction should normally finish quickly. OCR reports take longer.

The output stores the raw extracted text separately so parsing problems can be reviewed without reopening every PDF.


In [ ]:
import pandas as pd
from tqdm.auto import tqdm

records = []
failures = []
raw_text_records = []

for pdf_path in tqdm(pdf_paths, desc="Processing ECG PDFs"):
    relative_path = str(pdf_path.relative_to(extract_dir))

    try:
        file_hash = calculate_sha256(pdf_path)
        raw_text, method = extract_report_text(pdf_path)
        parsed = parse_ecg_report(raw_text, method)

        row = {
            "source_file": pdf_path.name,
            "relative_path": relative_path,
            "case_id": infer_case_id(pdf_path.name),
            "sha256": file_hash,
            **parsed
        }

        row = validate_row(row)
        records.append(row)

        raw_text_records.append({
            "source_file": pdf_path.name,
            "sha256": file_hash,
            "raw_text": raw_text
        })

    except Exception as error:
        failures.append({
            "source_file": pdf_path.name,
            "relative_path": relative_path,
            "error_type": type(error).__name__,
            "error_message": str(error)
        })

print("Successfully processed:", len(records))
print("Failed:", len(failures))


## 7. Create the raw dataset and inspect extraction quality

In [ ]:
dataset = pd.DataFrame(records)
failures_df = pd.DataFrame(failures)
raw_text_df = pd.DataFrame(raw_text_records)

if dataset.empty:
    raise RuntimeError("No reports were processed successfully.")

# Convert list columns to readable strings before saving.
dataset["machine_findings_text"] = dataset["machine_findings"].apply(
    lambda values: " | ".join(values) if isinstance(values, list) else ""
)
dataset["validation_warnings_text"] = dataset["validation_warnings"].apply(
    lambda values: " | ".join(values) if isinstance(values, list) else ""
)

print("Dataset shape:", dataset.shape)
display(dataset.head())

print("\nExtraction methods:")
display(dataset["extraction_method"].value_counts(dropna=False).to_frame("count"))

print("\nFeature completeness:")
display(
    dataset["feature_completeness_pct"]
    .describe()
    .to_frame("value")
)


## 8. Detect exact duplicates and repeated case IDs

Two kinds of repetition are tracked:

- **Exact duplicate:** identical file content and identical SHA-256 hash.
- **Repeated case ID:** filenames ending with the same numerical case identifier.

Repeated case IDs are grouped into the same dataset split to prevent nearly identical reports from leaking into both training and testing.


In [ ]:
dataset["is_exact_duplicate"] = dataset.duplicated(
    subset=["sha256"],
    keep="first"
)

dataset["case_id_report_count"] = dataset.groupby("case_id")[
    "source_file"
].transform("count")

dataset["is_repeated_case_id"] = dataset["case_id_report_count"] > 1

exact_duplicate_count = int(dataset["is_exact_duplicate"].sum())
repeated_case_rows = int(dataset["is_repeated_case_id"].sum())
unique_cases = dataset["case_id"].nunique()

print("Exact duplicate PDFs:", exact_duplicate_count)
print("Rows belonging to repeated case IDs:", repeated_case_rows)
print("Unique case IDs:", unique_cases)

display(
    dataset.loc[
        dataset["is_repeated_case_id"],
        ["source_file", "case_id", "primary_label", "sha256"]
    ].sort_values(["case_id", "source_file"]).head(30)
)


## 9. Build the cleaned dataset

In [ ]:
# Keep one copy of each exact PDF.
clean_dataset = dataset.loc[
    ~dataset["is_exact_duplicate"]
].copy().reset_index(drop=True)

# A row is eligible for model splitting only when all model features and
# a primary label are available.
model_ready_dataset = clean_dataset.loc[
    clean_dataset["is_model_ready"]
].copy().reset_index(drop=True)

incomplete_dataset = clean_dataset.loc[
    ~clean_dataset["is_model_ready"]
].copy().reset_index(drop=True)

print("All successfully parsed unique PDFs:", len(clean_dataset))
print("Model-ready rows:", len(model_ready_dataset))
print("Rows requiring review:", len(incomplete_dataset))


## 10. Inspect diagnosis distribution

The machine findings are preserved as multi-label text.

For the first baseline classifier, `primary_label` is provisionally defined as:

1. The first non-rhythm machine finding, or
2. The rhythm when no additional finding exists.

We will clean and consolidate these labels in Notebook 4 before final model training.


In [ ]:
label_counts = (
    model_ready_dataset["primary_label"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("primary_label")
    .reset_index(name="count")
)

display(label_counts.head(30))

print("Unique provisional primary labels:",
      model_ready_dataset["primary_label"].nunique())


## 11. Create leakage-safe train, validation, and test splits

The split uses **case IDs as groups**:

- Training: approximately 70%
- Validation: approximately 10%
- Testing: approximately 20%

All reports with the same case ID stay in the same split.

A fixed random seed makes the result reproducible.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

if len(model_ready_dataset) < 20:
    raise ValueError(
        "Too few model-ready rows for a meaningful split. "
        "Review the extraction results first."
    )

groups = model_ready_dataset["case_id"]

# First split: 70% train and 30% temporary.
first_split = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=RANDOM_STATE
)

train_indices, temporary_indices = next(
    first_split.split(model_ready_dataset, groups=groups)
)

train_dataset = model_ready_dataset.iloc[train_indices].copy()
temporary_dataset = model_ready_dataset.iloc[temporary_indices].copy()

# Second split: one third of temporary -> validation (~10% total),
# two thirds -> test (~20% total).
second_split = GroupShuffleSplit(
    n_splits=1,
    train_size=1/3,
    random_state=RANDOM_STATE
)

validation_local_indices, test_local_indices = next(
    second_split.split(
        temporary_dataset,
        groups=temporary_dataset["case_id"]
    )
)

validation_dataset = temporary_dataset.iloc[
    validation_local_indices
].copy()

test_dataset = temporary_dataset.iloc[
    test_local_indices
].copy()

train_dataset["dataset_split"] = "train"
validation_dataset["dataset_split"] = "validation"
test_dataset["dataset_split"] = "test"

split_dataset = pd.concat(
    [train_dataset, validation_dataset, test_dataset],
    ignore_index=True
)

print("Train rows:", len(train_dataset))
print("Validation rows:", len(validation_dataset))
print("Test rows:", len(test_dataset))

print("\nUnique case IDs:")
print("Train:", train_dataset["case_id"].nunique())
print("Validation:", validation_dataset["case_id"].nunique())
print("Test:", test_dataset["case_id"].nunique())

# Leakage checks.
train_cases = set(train_dataset["case_id"])
validation_cases = set(validation_dataset["case_id"])
test_cases = set(test_dataset["case_id"])

assert train_cases.isdisjoint(validation_cases)
assert train_cases.isdisjoint(test_cases)
assert validation_cases.isdisjoint(test_cases)

print("\nLeakage check passed: no case ID appears in multiple splits.")


## 12. Compare label distribution across splits

In [ ]:
split_label_distribution = pd.crosstab(
    split_dataset["primary_label"],
    split_dataset["dataset_split"]
)

split_label_distribution["total"] = split_label_distribution.sum(axis=1)
split_label_distribution = split_label_distribution.sort_values(
    "total",
    ascending=False
)

display(split_label_distribution.head(30))


## 13. Save all outputs

The notebook creates:

- `all_reports_dataset.csv`
- `model_ready_dataset.csv`
- `reports_requiring_review.csv`
- `train_dataset.csv`
- `validation_dataset.csv`
- `test_dataset.csv`
- `split_manifest.csv`
- `extraction_failures.csv`
- `raw_extracted_text.csv`
- `dataset_summary.json`
- `ecg_dataset_outputs.zip`


In [ ]:
import json
import shutil
from pathlib import Path

output_dir = Path("/content/ecg_dataset_outputs")

if output_dir.exists():
    shutil.rmtree(output_dir)

output_dir.mkdir(parents=True, exist_ok=True)

save_columns_to_drop = ["machine_findings", "validation_warnings"]

def save_csv(frame, filename):
    output = frame.drop(
        columns=save_columns_to_drop,
        errors="ignore"
    ).copy()
    output.to_csv(output_dir / filename, index=False)

save_csv(dataset, "all_reports_dataset.csv")
save_csv(model_ready_dataset, "model_ready_dataset.csv")
save_csv(incomplete_dataset, "reports_requiring_review.csv")
save_csv(train_dataset, "train_dataset.csv")
save_csv(validation_dataset, "validation_dataset.csv")
save_csv(test_dataset, "test_dataset.csv")

split_manifest = split_dataset[
    [
        "source_file",
        "relative_path",
        "case_id",
        "sha256",
        "primary_label",
        "dataset_split"
    ]
].copy()

split_manifest.to_csv(
    output_dir / "split_manifest.csv",
    index=False
)

failures_df.to_csv(
    output_dir / "extraction_failures.csv",
    index=False
)

raw_text_df.to_csv(
    output_dir / "raw_extracted_text.csv",
    index=False
)

summary = {
    "zip_file": zip_path.name,
    "pdf_files_found": len(pdf_paths),
    "successfully_processed": len(dataset),
    "failed_extractions": len(failures_df),
    "exact_duplicate_pdfs": exact_duplicate_count,
    "unique_pdf_records_after_duplicate_removal": len(clean_dataset),
    "unique_case_ids": int(clean_dataset["case_id"].nunique()),
    "model_ready_rows": len(model_ready_dataset),
    "reports_requiring_review": len(incomplete_dataset),
    "unique_primary_labels": int(
        model_ready_dataset["primary_label"].nunique()
    ),
    "train_rows": len(train_dataset),
    "validation_rows": len(validation_dataset),
    "test_rows": len(test_dataset),
    "random_state": RANDOM_STATE,
    "split_method": "GroupShuffleSplit grouped by case_id"
}

with open(
    output_dir / "dataset_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

archive_path = shutil.make_archive(
    "/content/ecg_dataset_outputs",
    "zip",
    output_dir
)

print(json.dumps(summary, indent=2))
print("\nZIP created:", archive_path)


## 14. Download the completed dataset package

In [ ]:
from google.colab import files

files.download("/content/ecg_dataset_outputs.zip")


## What to check after running this notebook

Before moving to Notebook 4, examine:

1. Number of successfully processed reports.
2. Number of reports requiring review.
3. Exact duplicates and repeated case IDs.
4. Provisional diagnosis label distribution.
5. Whether important classes have enough examples.
6. Whether any labels are spelling variations of the same condition.

Do not train the final classifier until the diagnosis labels have been cleaned and consolidated.
